Trabalho Final de Mineração de Dados
Resumo: Estudo de caso aplicado em uma base de dados real do setor de saúde. O objetivo é utilizar o processo de KDD (Knowledge Discovery in Databases) para a identificação de padrões, predição e extração de conhecimento sobre patologias cardíacas em crianças e jovens.
Aplicando os processos de Seleção de dados alvo, pré-processamento e limpeza, transformação, mineração de dados e Avaliação.

1° parte: analisar o que temos dentro do dataset e as variações de dados 

In [ ]:
import pandas as pd

df = pd.read_csv('UCMF_raw.csv')

# Loop para analisar cada coluna separadamente de forma isolada
for coluna in df.columns:
    print("\n" + "="*70)
    print(f"  ANÁLISE DE CADA VARIÁVEL: {coluna}")
    print("="*70)
    
    # 1. Informações básicas de tipo e preenchimento
    tipo = df[coluna].dtype
    total_linhas = len(df)
    nulos = df[coluna].isnull().sum()
    
    print(f"• Tipo de dado na coluna: {tipo}")
    print(f"• Linhas preenchidas: {df[coluna].count()} / {total_linhas}")
    print(f"• Linhas vazias (nulas): {nulos} ({ (nulos/total_linhas)*100 :.2f}%)")
    
    # 2. Extração de todos os valores diferentes (Únicos)
    # .dropna() remove os nulos temporariamente apenas para listar os valores reais existentes
    valores_distintos = df[coluna].dropna().unique()
    qtd_distintos = len(valores_distintos)
    
    print(f"• Quantidade de valores distintos/diferentes nesta coluna: {qtd_distintos}")
    print("-" * 70)
    
    # CASO A: Colunas com poucos valores diferentes (Ex: SEXO, SOPRO, B2, NORMAL X ANORMAL)
    if qtd_distintos <= 35:
        print("-> LISTA DE TODOS OS VALORES DIFERENTES PRESENTES:")
        print(valores_distintos)
        
        print("\n-> DISTRIBUIÇÃO E FREQUÊNCIA DE CADA VALOR (Incluindo Nulos):")
        frequencia = df[coluna].value_counts(dropna=False)
        percentual = df[coluna].value_counts(dropna=False, normalize=True) * 100
        
        tabela_separada = pd.DataFrame({
            'Quantidade (Qtd)': frequencia,
            'Proporção (%)': percentual.round(2)
        })
        print(tabela_separada)
        
    # CASO B: Colunas contínuas ou identificadores com muitos valores (Ex: ID, IMC, IDADE, FC)
    else:
        print(f"Nota: Como esta coluna possui muitos valores diferentes ({qtd_distintos}),")
        print("exibir todos criaria uma lista muito longa. Veja os 10 primeiros exemplos de valores únicos:")
        print(valores_distintos[:10])
        
        # Se for numérico, nos mostra as fronteiras (mínimo e máximo)
        if pd.api.types.is_numeric_dtype(df[coluna]):
            print("\n-> RESUMO DOS EXTREMOS NUMÉRICOS:")
            print(f"   - Menor valor real encontrado: {df[coluna].min()}")
            print(f"   - Maior valor real encontrado: {df[coluna].max()}")
            print(f"   - Média geral da coluna: {df[coluna].mean():.2f}")
            print(f"   - Mediana (valor do meio): {df[coluna].median():.2f}")
            
    print("\n" + "."*70) # Divisor de fim de bloco de coluna


  ANÁLISE ISOLADA DA VARIÁVEL: ID
• Tipo de dado na coluna: int64
• Linhas preenchidas: 12873 / 12873
• Linhas vazias (nulas): 0 (0.00%)
• Quantidade de valores distintos/diferentes nesta coluna: 12873
----------------------------------------------------------------------
Nota: Como esta coluna possui muitos valores diferentes (12873),
exibir todos criaria uma lista muito longa. Veja os 10 primeiros exemplos de valores únicos:
[ 1  2  3  4  7  8  9 10 11 12]

-> RESUMO DOS EXTREMOS NUMÉRICOS:
   - Menor valor real encontrado: 1
   - Maior valor real encontrado: 17873
   - Média geral da coluna: 9196.41
   - Mediana (valor do meio): 9222.00

......................................................................

  ANÁLISE ISOLADA DA VARIÁVEL: Peso
• Tipo de dado na coluna: float64
• Linhas preenchidas: 12555 / 12873
• Linhas vazias (nulas): 318 (2.47%)
• Quantidade de valores distintos/diferentes nesta coluna: 627
----------------------------------------------------------------------
N

O que encontramos após essa análise inicial?
- ID: Variável irrelevante para o processo;
- Peso: Possui dados faltantes (2.47%) e possui peso negativo, o que não é possível;
- Altura: Possui altura nula, o que não é possível;
- IMC: Possui dados faltantes (36.72%), caso tenhamos peso e altura, é possível calcular, menor valor foi 0;
- Atendimento: Pode ser relevante caso idade do paciente não foi coletado, dados faltantes (7.64%);
- Data de nascimento: Possui dados faltantes (10.69%), pode ser prejudicial pois o estudo é por faixa etária;
- Idade: Possui dados faltantes (11.62%), e com valores negativos, o que não é possível;
- Convênio: Variável irrelevante para o processo;
- Pulsos: Possui dados faltantes (9.28%), com palavras igual escritas de formas diferentes;
- PA sistólica: Possui muitos dados faltantes (60.05%), pode atrapalhar o processo;
- PA diastólica: Possui muitos dados faltantes (60.13%), pode atrapalhar o processo;
- PPA: Pode substituir os PAs, mas possui dados faltantes (1.69%), não calculado (70.54%) e sem valor (11.62%);
- Patologia: Possui dados faltantes (9.07%), e palavras iguais escritas de formas diferentes;
- B2: Possui dados faltantes (9.15%);
- Sopro: Possui dados faltantes (9.07%) e palavras iguais escritas de formas diferentes;
- Frequencia cardíaca: Possui dados faltantes (14.51%);
- Histórico de Doença 1: Possui dados faltantes (33.17%);
- Histórico de Doença 2: Possui dados faltantes (96.84%);
- Sexo: Poucos dados faltantes (0.03%), mas palavras iguais escritas de formas diferentes;
- Motivo 1: Possui dados faltantes (8.20%);
- Motivo 2: Possui dados faltantes (27.16%);

2° parte: Tratar os dados "sujos" para futuramente selecionar

Remover as variáveis irrelevantes inicialmente para o modelo (ID e convênio);
Tratar variáveis com valores fora do normal (Peso, Altura e idade);
Normalizar dados iguais com escritas diferentes que atrapalham na análise (Pulsos, Patologia, Sopro e Sexo);

In [1]:
import pandas as pd
import numpy as np

# ==========================================
# CONFIGURAÇÕES
# ==========================================

# Colunas que saem antes de qualquer análise (irrelevantes para o modelo)
COLS_REMOVER_SEMPRE = ['Peso', 'Altura']   # removidas após gerar IMC
COLS_REMOVER_ANTES  = ['ID', 'Convenio']   # irrelevantes para o modelo

ROTULO_NAO_INFORMADO = 'NÃO INFORMADO'


# ==========================================
# 1. FUNÇÕES DE LIMPEZA
# ==========================================

def normalizar_texto_categoria(serie, rotulo_nulo=ROTULO_NAO_INFORMADO):
    """Preenche nulos e padroniza maiúsculas/espaços.

    NÃO agrupa categorias raras em 'OUTRO' aqui — isso exigiria dados do
    holdout e causaria vazamento de informação.
    """
    return serie.fillna(rotulo_nulo).str.strip().str.upper()


def parse_valor_com_faixa(serie):
    """Converte valores no formato 'min-max' (ex: '92-100') para a média
    da faixa. Valores já numéricos passam direto; NaN permanece NaN.
    """
    serie_str = serie.astype(str)
    extraido  = serie_str.str.extract(r'^\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*$')
    media_faixa    = extraido.astype(float).mean(axis=1)
    valor_numerico = pd.to_numeric(serie, errors='coerce')
    return valor_numerico.fillna(media_faixa)


def carregar_e_limpar_dados(filepath: str) -> pd.DataFrame:
    """
    Lê o CSV e aplica todas as limpezas na seguinte ordem:
      1. Strip de espaços em colunas texto
      2. Remoção de colunas irrelevantes (ID, Convenio)
      3. Normalização de NORMAL X ANORMAL, SEXO, SOPRO, PULSOS, PPA, Patologia
      4. Limites clínicos em IDADE, PA, FC e recálculo de IMC
      5. Remoção de duplicatas
    """
    df = pd.read_csv(filepath)

    # ── 1. Strip geral em colunas texto ─────────────────────────────────
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip()

    # ── 2. Remove colunas irrelevantes para o modelo ─────────────────────
    df = df.drop(columns=COLS_REMOVER_ANTES, errors='ignore')

    # ── 3. Normalização de variáveis categóricas ─────────────────────────

    # NORMAL X ANORMAL — unifica grafias ('Normais' → 'Normal')
    if 'NORMAL X ANORMAL' in df.columns:
        df['NORMAL X ANORMAL'] = (
            df['NORMAL X ANORMAL']
            .str.title()
            .replace('Normais', 'Normal')
        )

    # SEXO — mantém só M/F; 'Indeterminado' ou qualquer inicial diferente → NaN
    # (registros com SEXO indeterminado têm padrão de incompletude global,
    # indicativo de erro de cadastro, não de categoria clínica real)
    if 'SEXO' in df.columns:
        df['SEXO'] = (
            df['SEXO']
            .str.upper()
            .str[0]
            .replace('I', np.nan)
        )

    # SOPRO — unifica capitalização ('sistólico' → 'Sistólico', etc.)
    if 'SOPRO' in df.columns:
        df['SOPRO'] = df['SOPRO'].replace({
            'sistólico': 'Sistólico',
            'contínuo':  'Contínuo',
        })

    # PULSOS — unifica capitalização ('NORMAIS' → 'Normais', etc.)
    if 'PULSOS' in df.columns:
        df['PULSOS'] = df['PULSOS'].replace({
            'NORMAIS': 'Normais',
            'AMPLOS':  'Amplos',
        })

    # PATOLOGIA — padroniza texto (strip + upper + preenche nulo)
    if 'Patologia' in df.columns:
        df['Patologia'] = normalizar_texto_categoria(df['Patologia'])

    # PPA — '#VALUE!' é erro de planilha, não resultado clínico → NaN
    # 'Não Calculado' é resultado real e permanece como categoria válida
    if 'PPA' in df.columns:
        df['PPA'] = df['PPA'].replace('#VALUE!', np.nan)

    # ── 4. Limites clínicos ───────────────────────────────────────────────

    # IDADE: faixa pediátrica esperada 0–19 anos
    if 'IDADE' in df.columns:
        df['IDADE'] = pd.to_numeric(df['IDADE'], errors='coerce')
        df.loc[(df['IDADE'] < 0) | (df['IDADE'] > 19), 'IDADE'] = np.nan

    # Peso e Altura (usados só para recalcular IMC)
    if 'Peso' in df.columns:
        df['Peso'] = pd.to_numeric(df['Peso'], errors='coerce')
        df.loc[df['Peso'] <= 0, 'Peso'] = np.nan

    if 'Altura' in df.columns:
        df['Altura'] = pd.to_numeric(df['Altura'], errors='coerce')
        df.loc[df['Altura'] <= 0, 'Altura'] = np.nan

    # IMC recalculado a partir de Peso/Altura já limpos
    if 'Peso' in df.columns and 'Altura' in df.columns:
        altura_m = df['Altura'] / 100
        df['IMC'] = np.where(altura_m > 0, df['Peso'] / (altura_m ** 2), np.nan)
        df.loc[(df['IMC'] < 5) | (df['IMC'] > 45), 'IMC'] = np.nan
        df = df.drop(columns=COLS_REMOVER_SEMPRE, errors='ignore')

    # PA SISTÓLICA
    if 'PA SISTOLICA' in df.columns:
        df['PA SISTOLICA'] = pd.to_numeric(df['PA SISTOLICA'], errors='coerce')
        df.loc[
            (df['PA SISTOLICA'] <= 0) | (df['PA SISTOLICA'] > 300),
            'PA SISTOLICA'
        ] = np.nan

    # PA DIASTÓLICA
    if 'PA DIASTOLICA' in df.columns:
        df['PA DIASTOLICA'] = pd.to_numeric(df['PA DIASTOLICA'], errors='coerce')
        df.loc[
            (df['PA DIASTOLICA'] <= 0) | (df['PA DIASTOLICA'] > 200),
            'PA DIASTOLICA'
        ] = np.nan

    # FC: suporta intervalos como '92-100' → média da faixa
    if 'FC' in df.columns:
        df['FC'] = parse_valor_com_faixa(df['FC'])
        df.loc[(df['FC'] < 30) | (df['FC'] > 250), 'FC'] = np.nan

    # ── 5. Remove duplicatas (exceto coluna ID, já removida) ─────────────
    n_antes = len(df)
    df = df.drop_duplicates().copy()
    n_removidas = n_antes - len(df)
    if n_removidas:
        print(f"[AVISO] {n_removidas} linha(s) duplicada(s) removida(s).")

    return df


# ==========================================
# 2. RELATÓRIO DE ANÁLISE POR VARIÁVEL
# ==========================================

def gerar_relatorio(df: pd.DataFrame) -> None:
    """Imprime o mesmo relatório coluna a coluna do script original,
    mas agora sobre o DataFrame já limpo."""

    print("\n" + "=" * 70)
    print("  RELATÓRIO DE ANÁLISE — DADOS APÓS LIMPEZA")
    print(f"  Total de linhas: {len(df)}  |  Total de colunas: {len(df.columns)}")
    print("=" * 70)

    for coluna in df.columns:
        print("\n" + "=" * 70)
        print(f"  ANÁLISE DE CADA VARIÁVEL: {coluna}")
        print("=" * 70)

        tipo        = df[coluna].dtype
        total_linhas = len(df)
        nulos       = df[coluna].isnull().sum()

        print(f"• Tipo de dado na coluna      : {tipo}")
        print(f"• Linhas preenchidas          : {df[coluna].count()} / {total_linhas}")
        print(f"• Linhas vazias (nulas)       : {nulos} ({(nulos / total_linhas) * 100:.2f}%)")

        # Valores distintos (excluindo nulos para listar só reais)
        valores_distintos = df[coluna].dropna().unique()
        qtd_distintos     = len(valores_distintos)

        print(f"• Qtd de valores distintos    : {qtd_distintos}")
        print("-" * 70)

        # ── CASO A: poucas categorias ─────────────────────────────────────
        if qtd_distintos <= 35:
            print("-> LISTA DE TODOS OS VALORES DIFERENTES PRESENTES:")
            print(valores_distintos)

            print("\n-> DISTRIBUIÇÃO E FREQUÊNCIA DE CADA VALOR (Incluindo Nulos):")
            frequencia  = df[coluna].value_counts(dropna=False)
            percentual  = df[coluna].value_counts(dropna=False, normalize=True) * 100
            tabela = pd.DataFrame({
                'Quantidade (Qtd)': frequencia,
                'Proporção (%)':    percentual.round(2)
            })
            print(tabela)

        # ── CASO B: muitos valores (contínuos / identificadores) ──────────
        else:
            print(f"Nota: coluna com muitos valores distintos ({qtd_distintos}).")
            print("10 primeiros exemplos de valores únicos:")
            print(valores_distintos[:10])

            if pd.api.types.is_numeric_dtype(df[coluna]):
                print("\n-> RESUMO DOS EXTREMOS NUMÉRICOS:")
                print(f"   - Menor valor real encontrado : {df[coluna].min()}")
                print(f"   - Maior valor real encontrado : {df[coluna].max()}")
                print(f"   - Média geral da coluna       : {df[coluna].mean():.2f}")
                print(f"   - Mediana (valor do meio)     : {df[coluna].median():.2f}")

        print("\n" + "." * 70)


# ==========================================
# 3. EXECUÇÃO
# ==========================================

if __name__ == "__main__":
    ARQUIVO = 'UCMF_raw.csv'          # ← ajuste o caminho se necessário

    print(f"\nCarregando e limpando '{ARQUIVO}'...")
    df_limpo = carregar_e_limpar_dados(ARQUIVO)
    print(f"Dataset pronto: {len(df_limpo)} linhas × {len(df_limpo.columns)} colunas.\n")

    gerar_relatorio(df_limpo)


Carregando e limpando 'UCMF_raw.csv'...
[AVISO] 999 linha(s) duplicada(s) removida(s).
Dataset pronto: 11874 linhas × 17 colunas.


  RELATÓRIO DE ANÁLISE — DADOS APÓS LIMPEZA
  Total de linhas: 11874  |  Total de colunas: 17

  ANÁLISE DE CADA VARIÁVEL: IMC
• Tipo de dado na coluna      : float64
• Linhas preenchidas          : 7947 / 11874
• Linhas vazias (nulas)       : 3927 (33.07%)
• Qtd de valores distintos    : 3482
----------------------------------------------------------------------
Nota: coluna com muitos valores distintos (3482).
10 primeiros exemplos de valores únicos:
[19.22337562 14.         19.17159763 17.54309022 20.45437943 21.49959688
 15.69713758 16.46062886 18.74219075 13.88394446]

-> RESUMO DOS EXTREMOS NUMÉRICOS:
   - Menor valor real encontrado : 5.859374999999999
   - Maior valor real encontrado : 44.61629982153481
   - Média geral da coluna       : 17.33
   - Mediana (valor do meio)     : 16.74

...............................................................

/tmp/ipykernel_13581/3949162296.py:51: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:
